## Exploratory Data Analysis (EDA)

This notebook explores daily Beijing air-quality and ERA5 weather summaries from the MySQL `air_quality.aqi_daily` table. The table contains one record per station and date across the available 2013–2025 data.

The analysis covers PM2.5, PM10, SO2, NO2, CO, and O3; compares daily and monthly patterns across stations; and examines relationships among pollutant and weather summaries. Rerun the notebook after loading `aqi_daily` so the charts and statistics reflect the current table.


## 1. Import Libraries

The required Python libraries are imported for data manipulation, visualization, and database connectivity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

plt.style.use("seaborn-v0_8-whitegrid")

## 2. Connect to Database

Connect to the MySQL `air_quality` database and load the daily station records from `aqi_daily`.


In [ ]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")

    df = pd.read_sql_query("SELECT * FROM aqi_daily", engine)
    combined_df = df.copy()
    print(f"Loaded {len(df):,} daily station rows from aqi_daily.")
except Exception as error:
    raise RuntimeError(
        "Could not connect to MySQL or load the aqi_daily table. "
        "Check that MySQL is running and air_quality.aqi_daily exists."
    ) from error

In [ ]:
combined_df.head()


In [ ]:
missing_values = combined_df.isnull().sum()

print(missing_values)

## 3. Define Pollutants and Weather Features

The daily table contains the six pollutant summaries and daily ERA5 weather features. The analysis uses the available daily columns directly.


In [ ]:
pollutants = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
]

weather_measurements = [
    "TEMP_mean",
    "TEMP_range",
    "PRES_mean",
    "DEWP_mean",
    "RH_mean",
    "RAIN_sum",
    "WSPM_mean",
    "wd_sin",
    "wd_cos",
]

analysis_columns = pollutants + weather_measurements
missing_columns = [column for column in analysis_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"aqi_daily is missing analysis columns: {missing_columns}")
df[analysis_columns] = df[analysis_columns].apply(pd.to_numeric, errors="coerce")

print("Pollutants included in the analysis:")
print(pollutants)
print("Daily weather features included in the analysis:")
print(weather_measurements)

pollutant_statistics = df[pollutants].describe()
pollutant_statistics

## 4. Graph 1: Average Pollutant Concentration

The average concentration of each pollutant is calculated for every monitoring station. The visualization allows the pollutant levels to be compared across the 12 stations.

Separate graphs are used for each pollutant because the pollutants have different measurement scales and units. This makes the differences between stations easier to interpret.

In [ ]:
station_averages = combined_df.groupby("station")[pollutants].mean()

station_averages

for pollutant in pollutants:

    ax = station_averages[pollutant].plot(
        kind="bar",
        figsize=(14, 6)
    )

    ax.set_title(
        f"Average {pollutant} Concentration by Monitoring Station"
    )

    ax.set_xlabel("Monitoring Station")
    ax.set_ylabel("Average Concentration")
    ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

In [ ]:
for pollutant in pollutants:

    highest_station = station_averages[pollutant].idxmax()
    highest_value = station_averages[pollutant].max()

    lowest_station = station_averages[pollutant].idxmin()
    lowest_value = station_averages[pollutant].min()

    overall_average = station_averages[pollutant].mean()

    print(f"\n{pollutant}")
    print("=" * 50)

    print(
        f"Overall average concentration : "
        f"{overall_average:.2f}"
    )

    print(
        f"Highest average station       : "
        f"{highest_station}"
    )

    print(
        f"Highest average concentration : "
        f"{highest_value:.2f}"
    )

    print(
        f"Lowest average station        : "
        f"{lowest_station}"
    )

    print(
        f"Lowest average concentration  : "
        f"{lowest_value:.2f}"
    )

### Analysis

The graph shows how the average concentration of the selected pollutant differs among the 12 monitoring stations. Differences between stations may indicate variations in local pollution conditions and surrounding environmental factors.

The station with the highest average value represents the location with the highest recorded average concentration for that pollutant, while the station with the lowest value represents the lowest average concentration.

## 5. Graph 2: Daily Average of Pollutants from across 12 monitoring station

Daily averages are calculated for the six pollutants using observations from the 12 monitoring stations. The graphs show how pollutant concentrations change from day to day throughout the study period.

This visualization helps identify fluctuations, periods of increased concentration, and general changes in pollutant levels over time.

In [ ]:
combined_df["date"] = pd.to_datetime(combined_df["date"], errors="coerce")
if combined_df["date"].isna().any():
    raise ValueError("aqi_daily contains invalid dates.")

daily_pollutants = (
    combined_df
    .groupby("date")[pollutants]
    .mean()
)

daily_pollutants.head()

In [ ]:
for pollutant in pollutants:
    plt.figure(figsize=(15, 6))
    
    plt.plot(
        daily_pollutants.index,
        daily_pollutants[pollutant]
    )
    
    plt.title(f"Daily Average {pollutant} Across 12 Monitoring Stations")
    plt.xlabel("Date")
    plt.ylabel("Average Concentration")
    plt.tight_layout()
    plt.show()

In [ ]:
for pollutant in pollutants:

    daily_station = (
        combined_df
        .groupby(["date", "station"])[pollutants]
        .mean()
    )

    daily_values = daily_station[pollutant].dropna()

    highest_value = daily_values.max()
    highest_date, highest_station = daily_values.idxmax()

    lowest_value = daily_values.min()
    lowest_date, lowest_station = daily_values.idxmin()

    print(f"\n{pollutant}")
    print("=" * 50)

    print(
        f"Highest daily concentration : {highest_value:.2f}"
    )
    print(
        f"Station with highest value  : {highest_station}"
    )
    print(
        f"Date of highest value       : {highest_date.date()}"
    )

    print(
        f"Lowest daily concentration  : {lowest_value:.2f}"
    )
    print(
        f"Station with lowest value   : {lowest_station}"
    )
    print(
        f"Date of lowest value        : {lowest_date.date()}"
    )

### Analysis

The daily trend shows that pollutant concentrations went up and down throughout the study period. Peaks indicate periods when the average concentration increased, while lower values indicate periods of reduced concentration.

The patterns may be influenced by factors such as weather conditions, seasonal changes, and local pollution sources.

## 6. Graph 3: Monthly Average of Pollutants Across 12 Monitoring Stations

Monthly averages are calculated for the six pollutants using all observations from the 12 monitoring stations. The graphs show the seasonal pattern of each pollutant throughout the study period.

This helps identify long-term monthly changes and whether pollution levels rise in specific seasons.

In [ ]:
monthly_pollutants = (
    combined_df
    .groupby(combined_df["date"].dt.to_period("M"))[pollutants]
    .mean()
)

monthly_pollutants.index = monthly_pollutants.index.astype(str)

monthly_pollutants.head()

In [ ]:
for pollutant in pollutants:
    plt.figure(figsize=(15, 6))

    plt.plot(
        monthly_pollutants.index,
        monthly_pollutants[pollutant],
        marker="o",
        linewidth=2
    )

    plt.title(f"Monthly Average {pollutant} Across 12 Monitoring Stations")
    plt.xlabel("Month")
    plt.ylabel("Average Concentration")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
for pollutant in pollutants:

    monthly_station = (
        combined_df
        .groupby([combined_df["date"].dt.to_period("M"), "station"])[pollutants]
        .mean()
    )

    monthly_values = monthly_station[pollutant].dropna()

    highest_value = monthly_values.max()
    highest_month, highest_station = monthly_values.idxmax()

    lowest_value = monthly_values.min()
    lowest_month, lowest_station = monthly_values.idxmin()

    print(f"\n{pollutant}")
    print("=" * 50)

    print(
        f"Highest monthly concentration : {highest_value:.2f}"
    )
    print(
        f"Station with highest value    : {highest_station}"
    )
    print(
        f"Month of highest value       : {highest_month.to_timestamp().strftime('%Y-%m')}"
    )

    print(
        f"Lowest monthly concentration  : {lowest_value:.2f}"
    )
    print(
        f"Station with lowest value     : {lowest_station}"
    )
    print(
        f"Month of lowest value        : {lowest_month.to_timestamp().strftime('%Y-%m')}"
    )

### Analysis

The monthly trend shows that pollutant concentrations changed in a seasonal pattern throughout the study period. Higher monthly values suggest periods of stronger pollution buildup, while lower values indicate months with improved air quality.

These patterns may reflect seasonality, weather conditions, and changes in emissions across the year. Examining monthly averages helps identify whether pollution is consistently elevated during certain months rather than only on specific days.

## Pollutants and Weather Measurements

This compares the linear relationships among pollutant concentrations and weather measurements.

In [ ]:
pearson_corr = df[analysis_columns].corr(method="pearson")
print(pearson_corr.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(
    pearson_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title("Pollutants and Weather Measurements")
plt.tight_layout()
plt.show()

### Interpretation

Use the correlation matrix above to see which daily pollutant and weather features move together. Correlation describes association in this dataset and does not establish that one measurement causes another.


## Ranked Pollutants and Weather Measurements

This compares the ranked relationships among pollutant concentrations and weather measurements.

In [ ]:
spearman_corr = df[analysis_columns].corr(method="spearman")
print(spearman_corr.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(
    spearman_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title("Ranked Relationships Among Pollutants and Weather Measurements")
plt.tight_layout()
plt.show()

### Interpretation

The Spearman matrix shows rank-based associations among daily pollutant and weather measurements. Compare it with the Pearson matrix; neither correlation measure establishes causation.


## Relationships between pollutans and weather measurements

This identifies the strongest relationships among pollutants and weather measurements.

In [ ]:
upper_triangle = np.triu_indices_from(pearson_corr, k=1)
strongest_pairs = pd.DataFrame({
    "variable_1": pearson_corr.index[upper_triangle[0]],
    "variable_2": pearson_corr.columns[upper_triangle[1]],
    "correlation": pearson_corr.to_numpy()[upper_triangle],
})
strongest_pairs["absolute_correlation"] = strongest_pairs["correlation"].abs()
strongest_pairs = strongest_pairs.sort_values(
    "absolute_correlation", ascending=False
).head(10)
strongest_pairs["relationship"] = (
    strongest_pairs["variable_1"] + " and " + strongest_pairs["variable_2"]
)
print(
    strongest_pairs[["variable_1", "variable_2", "correlation"]]
    .round(3)
    .to_string(index=False)
)

plt.figure(figsize=(10, 6))
colors = [
    "steelblue" if value >= 0 else "indianred"
    for value in strongest_pairs["correlation"]
]
plt.barh(
    strongest_pairs["relationship"][::-1],
    strongest_pairs["correlation"][::-1],
    color=colors[::-1],
)
plt.axvline(0, color="black", linewidth=0.8)
for index, value in enumerate(strongest_pairs["correlation"][::-1]):
    offset = 0.02 if value >= 0 else -0.02
    alignment = "left" if value >= 0 else "right"
    plt.text(
        value + offset,
        index,
        f"{value:.2f}",
        va="center",
        ha=alignment,
    )
plt.xlabel("Pearson correlation")
plt.title("Strongest Relationships")
plt.xlim(-1, 1)
plt.tight_layout()
plt.show()

### Interpretation

This chart highlights the strongest absolute Pearson correlations in the current `aqi_daily` data. Check the plotted values after rerunning; the leading relationships can change with the date range and available records.


## PM2.5 and Other Pollutants by Station

This compares PM2.5 with the other pollutants separately for each monitoring station.

In [ ]:
station_corr = (
    df.groupby("station")[pollutants]
    .corr()["PM2.5"]
    .unstack(level=1)
    .drop(columns="PM2.5")
)
print(station_corr.round(3))

plt.figure(figsize=(9, 7))
sns.heatmap(
    station_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
)
plt.title("PM2.5 Relationships with Other Pollutants by Station")
plt.tight_layout()
plt.show()

### Interpretation

Compare the station-specific PM2.5 correlations shown above. Differences indicate that pollutant relationships vary across stations; they should be interpreted as associations rather than causes.
